# 06 · Control de la detección — ¿podemos creerle?

**Qué hace:** somete el resultado del notebook 05 a cuatro pruebas que no necesitan datos de
referencia externos. Si alguna falla, hay que volver atrás antes de construir nada encima.

**Qué NO hace:** no valida la exactitud. Eso exige puntos interpretados a mano y muestreo
estratificado, y es el notebook 09. Aquí se comprueba **coherencia interna**: que el método no esté
detectando de más, que los parámetros no estén sobresegmentando, y que el año flojo de la serie no
esté distorsionando el resto.

Son cosas distintas y conviene no confundirlas. Un resultado puede pasar todos estos controles y
aun así ser inexacto; pero si **no** los pasa, es inútil validarlo.

### Qué consume

| Entrada | De dónde |
|---|---|
| Resultado de LandTrendr | Asset `cusco_05_resultado` (notebook 05) |
| Serie anual | Asset `cusco_04_serie_anual` (notebook 04) |
| Máscaras | Asset `cusco_02_mascara` (notebook 02) |
| Área de estudio | Asset `cusco_01_aoi` (notebook 01) |

### Qué produce

| Salida | Para qué |
|---|---|
| `outputs/06_control_deteccion/sensibilidad_maxsegments.csv` | Justificar la elección del parámetro en el informe |
| `outputs/06_control_deteccion/estabilidad_2025.csv` | Documentar la decisión sobre el año terminal |
| `outputs/06_control_deteccion/distribucion_rmse.csv` | Soporte de la banda de confianza |

Los tres se **guardan**, no solo se imprimen. Si dentro de tres meses alguien pregunta por qué
`maxSegments` es 4, la respuesta tiene que estar en un archivo y no en una celda que ya se borró.

---

## Los cuatro controles

**1 · ¿Detecta de más?** Se compara la tasa de detección **dentro** del pajonal contra **fuera**. Si
el método encuentra "degradación" con la misma frecuencia sobre zona urbana o cultivos que sobre
pajonal, no está detectando degradación de pastizal sino algo genérico — ruido, artefactos, o un
umbral demasiado bajo. Es el control más informativo del notebook y no cuesta nada, porque el
notebook 05 corrió sobre la provincia entera a propósito (su decisión 1.4).

**2 · ¿Está sobresegmentando?** Se corre con `maxSegments` 3, 4 y 5 y se mira cómo cambia la
detección dentro y fuera. Si al subir el parámetro la detección crece **más afuera que adentro**, el
algoritmo está encontrando quiebres en terreno que no tiene por qué estar cambiando: la señal de que
el valor es demasiado alto para una serie de 13 años.

**3 · ¿2025 distorsiona?** El notebook 04 dejó 2025 marcado como el año débil: 2.94 observaciones
limpias de media y solo el 63 % de la provincia alcanzando el mínimo. Y es el **año terminal**, donde
los vértices son menos confiables porque no hay observaciones posteriores que confirmen el cambio. Un
valor sesgado ahí tiene palanca sobre la pendiente del último segmento y puede fabricar una señal de
degradación reciente. Se corre con y sin 2025 y se compara si los quiebres **anteriores** coinciden.

**4 · ¿El ajuste es creíble?** Distribución del RMSE y trayectorias de muestra con la serie original
y la ajustada superpuestas. Es la única forma de ver si la polilínea sigue la forma real de la serie
o inventa quiebres donde solo hay ruido.

## Una nota sobre el costo

Los controles 2 y 3 **vuelven a correr LandTrendr** — no hay forma de probar otro `maxSegments` sin
segmentar otra vez. Lo que sí se evita es recalcular la serie: al leerla del asset, el medoide de 13
años ya está resuelto y lo único que se repite es la segmentación, que es la parte barata. Esa era la
causa real de los `Computed image is too large` en versiones anteriores.

**Orden del proyecto:** 01 área → 02 cobertura → 03 fuego → 04 serie anual → 05 detección →
**06 control** → 07 tipo de cambio → 08 clima → 09 validación → 10 entregables.

## 1 · Configuración

Los parámetros tienen que ser **los mismos** que usó el notebook 05, o los controles no dirían nada
sobre el resultado que se quiere controlar. Por eso se leen de los metadatos del asset en vez de
escribirse otra vez a mano.

In [ ]:
from pathlib import Path

import ee
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

GEE_PROJECT = "viirs-peru"

ASSET_AOI = f"projects/{GEE_PROJECT}/assets/cusco_01_aoi"
ASSET_MASCARA = f"projects/{GEE_PROJECT}/assets/cusco_02_mascara"
ASSET_SERIE = f"projects/{GEE_PROJECT}/assets/cusco_04_serie_anual"
ASSET_RESULTADO = f"projects/{GEE_PROJECT}/assets/cusco_05_resultado"

ANIO_INI, ANIO_FIN = 2013, 2025
ANIOS = list(range(ANIO_INI, ANIO_FIN + 1))

INDICE_PRIMARIO = "negTCW"
INDICES_FTV = ["NDVI", "MSAVI", "TCG", "TCB", "BSI", "NDMI"]
INDICES = [INDICE_PRIMARIO] + INDICES_FTV

FILAS_SEG = ["anio_ini", "anio_fin", "val_ini", "val_fin", "magnitud", "duracion", "tasa"]

MAXSEG_PRUEBA = [3, 4, 5]      # control 2
ANIO_TERMINAL = 2025           # control 3: el año que se pone a prueba
ANIO_CORTE_COMPARACION = 2023  # solo se comparan quiebres anteriores a este año

ESCALA = 30
ESCALA_STATS = 60
GRID_CRS = "EPSG:32719"
TILE_SCALE = 4


def buscar_raiz(inicio=None):
    inicio = Path(inicio or Path.cwd()).resolve()
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg").is_file():
            return carpeta
    raise FileNotFoundError("No encuentro data/raw/Provincia/PROVINCIA.gpkg desde esta ubicación.")


def iniciar_gee(proyecto=GEE_PROJECT):
    try:
        ee.Initialize(project=proyecto)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=proyecto)


def con_reintentos(fn, intentos=6, espera=5):
    import time
    for i in range(intentos):
        try:
            return fn()
        except Exception as e:
            msg = repr(e)
            if not any(t in msg for t in ["concurrent", "Too many", "Timed out", "timed out",
                                          "quota", "503", "500", "Connection"]) or i == intentos - 1:
                raise
            pausa = espera * (i + 1)
            print(f"\n    reintento {i + 1} en {pausa}s · {msg[:90]}")
            time.sleep(pausa)


def evaluar_serie(objetos, etiquetas=None, pausa=0.4):
    import time
    etiquetas = etiquetas or [str(i) for i in range(len(objetos))]
    salida = []
    for i, (obj, et) in enumerate(zip(objetos, etiquetas), 1):
        print(f"  [{i}/{len(objetos)}] {et}", end="  ", flush=True)
        salida.append(con_reintentos(lambda o=obj: o.getInfo()))
        print("ok", flush=True)
        if pausa and i < len(objetos):
            time.sleep(pausa)
    return salida


iniciar_gee()
RAIZ = buscar_raiz()
SALIDA = RAIZ / "outputs" / "06_control_deteccion"
SALIDA.mkdir(parents=True, exist_ok=True)
print("Raíz del proyecto:", RAIZ)
CONFIG_LISTA = True

In [ ]:
# --- Insumos, todos desde assets -------------------------------------------
faltan = []
try:
    aoi = ee.FeatureCollection(ASSET_AOI).geometry()
    print("✓ AOI")
except Exception as e:
    faltan.append(("cusco_01_aoi", "notebook 01", repr(e)[:80]))

try:
    mascaras = ee.Image(ASSET_MASCARA)
    pastizal = mascaras.select("pastizal")
    MASCARA_ANALISIS = mascaras.select("analisis")
    print("✓ Máscaras")
except Exception as e:
    faltan.append(("cusco_02_mascara", "notebook 02", repr(e)[:80]))

try:
    serie = ee.ImageCollection(ASSET_SERIE).map(lambda img: img.set(
        "year", ee.Date(ee.Number(img.get("system:time_start"))).get("year")))
    n_img = con_reintentos(lambda: serie.size().getInfo())
    print(f"✓ Serie anual · {n_img} imágenes")
except Exception as e:
    faltan.append(("cusco_04_serie_anual", "notebook 04", repr(e)[:80]))

try:
    RESULTADO = ee.Image(ASSET_RESULTADO)
    bandas_res = con_reintentos(lambda: RESULTADO.bandNames().getInfo())
    meta = con_reintentos(lambda: RESULTADO.toDictionary().getInfo())
    print(f"✓ Resultado   · {len(bandas_res)} bandas")
except Exception as e:
    faltan.append(("cusco_05_resultado", "notebook 05, sección 8", repr(e)[:80]))

if faltan:
    print("\n" + "=" * 72)
    print("FALTAN ASSETS")
    print("=" * 72)
    for nombre, donde, err in faltan:
        print(f"  {nombre}  →  generarlo en {donde}")
        print(f"     {err}")
    raise RuntimeError("Ejecuta las celdas de exportación indicadas y vuelve a empezar.")

# --- Parámetros, leídos de los metadatos del asset -------------------------
# No se re-escriben a mano: tienen que ser los MISMOS que usó el notebook 05, o los
# controles no dirían nada sobre el resultado que se quiere controlar.
def _num(v, defecto):
    try:
        return type(defecto)(v)
    except (TypeError, ValueError):
        return defecto


PARAMS = {
    "maxSegments": _num(meta.get("maxSegments"), 4),
    "spikeThreshold": _num(meta.get("spikeThreshold"), 0.9),
    "vertexCountOvershoot": _num(meta.get("vertexCountOvershoot"), 3),
    "preventOneYearRecovery": str(meta.get("preventOneYearRecovery", "True")) == "True",
    "recoveryThreshold": _num(meta.get("recoveryThreshold"), 0.25),
    "pvalThreshold": _num(meta.get("pvalThreshold"), 0.05),
    "bestModelProportion": _num(meta.get("bestModelProportion"), 0.75),
    "minObservationsNeeded": _num(meta.get("minObservationsNeeded"), 6),
}
UMBRAL_MAG = _num(meta.get("umbral_magnitud"), 0.0)
DUR_MIN = _num(meta.get("duracion_minima"), 3)

print("\nParámetros leídos del asset:")
for k, v in PARAMS.items():
    print(f"  {k:24s} {v}")
print(f"  {'umbral_magnitud':24s} {UMBRAL_MAG}")
print(f"  {'duracion_minima':24s} {DUR_MIN}")
print(f"  {'fecha_corrida':24s} {meta.get('fecha_corrida')}")

if UMBRAL_MAG == 0:
    print("\n⚠ El umbral no estaba en los metadatos del asset. Revisa que el notebook 05")
    print("  haya exportado con los metadatos completos.")


# --- Funciones compartidas con el 05 ---------------------------------------
def km2_de(mascara, escala=None):
    v = (ee.Image.pixelArea().divide(1e6).updateMask(mascara)
         .reduceRegion(ee.Reducer.sum(), aoi, escala or ESCALA_STATS,
                       maxPixels=1e10, bestEffort=True, tileScale=TILE_SCALE))
    return ee.Number(v.get("area"))


def correr_landtrendr(coleccion, params=None, anios=None):
    p = {**PARAMS, **(params or {})}
    col = coleccion.select(INDICES)
    if anios:
        col = col.filter(ee.Filter.inList("year", anios))
    return ee.Algorithms.TemporalSegmentation.LandTrendr(
        timeSeries=col.sort("system:time_start"), **p)


def datos_segmentos(lt_img):
    """Port de getSegmentData() de lt-gee."""
    arr = lt_img.select("LandTrendr")
    vertices = arr.arrayMask(arr.arraySlice(0, 3, 4))
    izq, der = vertices.arraySlice(1, 0, -1), vertices.arraySlice(1, 1)
    anio_ini, anio_fin = izq.arraySlice(0, 0, 1), der.arraySlice(0, 0, 1)
    val_ini, val_fin = izq.arraySlice(0, 2, 3), der.arraySlice(0, 2, 3)
    duracion = anio_fin.subtract(anio_ini)
    magnitud = val_fin.subtract(val_ini)
    return ee.Image.cat([anio_ini, anio_fin, val_ini, val_fin,
                         magnitud, duracion, magnitud.divide(duracion)]).toArray(0)


def atributo(arr, nombre):
    i = FILAS_SEG.index(nombre)
    return arr.arraySlice(0, i, i + 1).arrayProject([1]).arrayFlatten([[nombre]])


def mapa_cambio(arr, umbral, dur_min=None, dur_max=None):
    i_m, i_d = FILAS_SEG.index("magnitud"), FILAS_SEG.index("duracion")
    mag, dur = arr.arraySlice(0, i_m, i_m + 1), arr.arraySlice(0, i_d, i_d + 1)
    cumple = mag.gt(umbral)
    if dur_min is not None:
        cumple = cumple.And(dur.gte(dur_min))
    if dur_max is not None:
        cumple = cumple.And(dur.lte(dur_max))
    filtrado = arr.arrayMask(cumple)
    clave = filtrado.arraySlice(0, i_m, i_m + 1).toArray(0).multiply(-1)
    princ = filtrado.arraySort(clave).arraySlice(1, 0, 1)
    return ee.Image.cat([atributo(princ, n) for n in FILAS_SEG])


CALIDAD = RESULTADO.select("anios_buenos").gte(PARAMS["minObservationsNeeded"])
MASCARA_FINAL = MASCARA_ANALISIS.And(CALIDAD).selfMask()
FUERA = MASCARA_ANALISIS.unmask(0).Not()

grad = RESULTADO.select(FILAS_SEG)
abrupto = RESULTADO.select([f"ab_{n}" for n in FILAS_SEG]).rename(FILAS_SEG)
rmse = RESULTADO.select("rmse")
print("\nTodo listo. El resultado se lee del asset: no se vuelve a segmentar en este bloque.")

## 2 · Control 1 · ¿Detecta de más?

La comparación más informativa del notebook, y la más barata.

El notebook 05 corrió sobre la provincia entera a propósito, no solo sobre el pajonal. Eso permite
preguntar: **¿con qué frecuencia el método encuentra "degradación" fuera del pajonal?**

Fuera de la máscara hay ciudad, cultivos, roca, agua y glaciar. En ninguno de esos sitios tiene
sentido hablar de degradación de pastizal. Así que la detección ahí es, en su mayoría, ruido del
método: artefactos del compuesto, nubes mal enmascaradas, o un umbral demasiado permisivo.

**Cómo leerlo.** Lo que importa no son los km² absolutos —las dos áreas tienen tamaños distintos—
sino la **tasa**: qué porcentaje de cada área aparece como degradada.

- Tasa dentro **mucho mayor** que fuera → el método está captando algo específico del pajonal. Buena señal.
- Tasas **parecidas** → está detectando algo genérico que no distingue cobertura. Mala señal: revisar
  el umbral y la máscara antes de seguir.
- Tasa fuera **mayor** → algo está mal de raíz.

No hay un umbral canónico para "mucho mayor". Es un juicio, pero un factor de 2 o más a favor del
pajonal es lo que cabría esperar si el método funciona.

In [ ]:
hay_grad = grad.select("magnitud").gt(0).unmask(0)
hay_abr = abrupto.select("magnitud").gt(0).unmask(0)

print("Midiendo detección dentro y fuera del pajonal:")
vals = evaluar_serie(
    [km2_de(MASCARA_FINAL),
     km2_de(FUERA.selfMask()),
     km2_de(hay_grad.And(MASCARA_FINAL.unmask(0)).selfMask()),
     km2_de(hay_grad.And(FUERA).selfMask()),
     km2_de(hay_abr.And(MASCARA_FINAL.unmask(0)).selfMask()),
     km2_de(hay_abr.And(FUERA).selfMask())],
    ["área pajonal", "área fuera", "gradual dentro", "gradual fuera",
     "abrupta dentro", "abrupta fuera"])

km_dentro, km_fuera = float(vals[0]), float(vals[1])
control1 = pd.DataFrame([
    {"zona": "pajonal (máscara de análisis)", "area_km2": round(km_dentro, 2),
     "gradual_km2": round(vals[2], 3), "abrupta_km2": round(vals[4], 3)},
    {"zona": "fuera del pajonal", "area_km2": round(km_fuera, 2),
     "gradual_km2": round(vals[3], 3), "abrupta_km2": round(vals[5], 3)},
])
control1["tasa_gradual_%"] = (control1["gradual_km2"] / control1["area_km2"] * 100).round(2)
control1["tasa_abrupta_%"] = (control1["abrupta_km2"] / control1["area_km2"] * 100).round(2)
display(control1.set_index("zona"))

t_dentro = control1.loc[0, "tasa_gradual_%"]
t_fuera = control1.loc[1, "tasa_gradual_%"]
razon = t_dentro / t_fuera if t_fuera else float("inf")

print(f"\nTasa de degradación gradual · dentro {t_dentro:.2f} % · fuera {t_fuera:.2f} %")
print(f"Razón dentro/fuera: {razon:.2f}×")
print()
if razon >= 2:
    print("✓ La detección se concentra claramente en el pajonal.")
    print("  El método está captando algo específico de esa cobertura.")
elif razon >= 1.2:
    print("~ La detección se concentra en el pajonal, pero con poco margen.")
    print("  Revisar el umbral antes de dar la cifra por buena.")
else:
    print("✗ La detección NO distingue el pajonal del resto del paisaje.")
    print("  Probablemente se esté midiendo ruido del método y no degradación.")
    print("  Revisar: umbral de magnitud, máscara de calidad y enmascarado de nubes.")

control1.to_csv(SALIDA / "control_dentro_fuera.csv", index=False)
print("\nGuardado en", SALIDA / "control_dentro_fuera.csv")

## 3 · Control 2 · Sensibilidad a `maxSegments`

El default de LT-GEE es 6, pensado para bosque y series de 25–35 años. Con 13 observaciones anuales,
6 segmentos permiten 7 vértices: casi un quiebre por año. El riesgo es que el algoritmo deje de
describir la trayectoria y empiece a seguir el ruido climático interanual.

Se corren 3, 4 y 5 y se mira cómo cambia la detección **en los dos lados de la máscara**.

**Cómo leerlo.** Que la detección suba con `maxSegments` es esperable: más segmentos, más
oportunidades de encontrar un quiebre que pase el umbral. Lo que delata la sobresegmentación es que
suba **más afuera que adentro**, porque fuera del pajonal no hay degradación que encontrar: todo
incremento ahí es señal falsa. Por eso la columna que importa es la razón fuera/dentro.

Esta celda vuelve a segmentar tres veces. Es lo caro del notebook, pero la serie viene del asset, así
que el medoide de 13 años ya está resuelto y solo se repite la segmentación.

In [ ]:
print(f"Probando maxSegments = {MAXSEG_PRUEBA}")
filas = []
for ms in MAXSEG_PRUEBA:
    print(f"\n  maxSegments = {ms}")
    g = mapa_cambio(datos_segmentos(correr_landtrendr(serie, {"maxSegments": ms})),
                    UMBRAL_MAG, dur_min=DUR_MIN)
    hay = g.select("magnitud").gt(0).unmask(0)
    v = evaluar_serie(
        [km2_de(hay.And(MASCARA_FINAL.unmask(0)).selfMask()), km2_de(hay.And(FUERA).selfMask())],
        ["dentro", "fuera"])
    filas.append({"maxSegments": ms,
                  "km2_dentro": round(v[0], 3), "km2_fuera": round(v[1], 3)})

sens = pd.DataFrame(filas)
sens["tasa_dentro_%"] = (sens["km2_dentro"] / km_dentro * 100).round(2)
sens["tasa_fuera_%"] = (sens["km2_fuera"] / km_fuera * 100).round(2)
sens["razon_dentro_fuera"] = (sens["tasa_dentro_%"] / sens["tasa_fuera_%"]).round(2)
sens["en_uso"] = np.where(sens["maxSegments"] == PARAMS["maxSegments"], "<<<", "")
display(sens.set_index("maxSegments"))

sens.to_csv(SALIDA / "sensibilidad_maxsegments.csv", index=False)
print("Guardado en", SALIDA / "sensibilidad_maxsegments.csv")

fig, eje = plt.subplots(figsize=(7, 3.5))
eje.plot(sens["maxSegments"], sens["tasa_dentro_%"], "o-", color="#2e7d32",
         lw=2, label="dentro del pajonal")
eje.plot(sens["maxSegments"], sens["tasa_fuera_%"], "s--", color="#c62828",
         lw=2, label="fuera (debería ser bajo y plano)")
eje.set_xlabel("maxSegments")
eje.set_ylabel("% del área detectada")
eje.set_xticks(MAXSEG_PRUEBA)
eje.legend(fontsize=8)
eje.grid(alpha=0.3)
plt.title("Sensibilidad a maxSegments")
plt.tight_layout()
plt.show()

razones = sens.set_index("maxSegments")["razon_dentro_fuera"]
mejor = razones.idxmax()
print(f"Mejor razón dentro/fuera con maxSegments = {mejor} ({razones[mejor]:.2f}×)")
print(f"Valor en uso: {PARAMS['maxSegments']}")
if mejor != PARAMS["maxSegments"]:
    print(f"\n~ {mejor} discrimina mejor que {PARAMS['maxSegments']}. No es razón suficiente")
    print("  para cambiarlo por sí sola: la decisión definitiva se toma con los puntos")
    print("  interpretados del notebook 09, eligiendo por F1. Pero queda anotado.")
else:
    print("\n✓ El valor en uso es el que mejor discrimina de los tres probados.")

## 4 · Control 3 · ¿2025 distorsiona la segmentación?

El notebook 04 dejó 2025 marcado: 2.94 observaciones limpias de media y solo el 63 % de la provincia
alcanzando el mínimo de 3. Dos problemas se suman ahí:

1. **El medoide de 2025 se apoya a menudo en una sola escena**, así que cualquier bruma residual
   queda sin filtrar.
2. **Es el punto extremo de la serie.** Un valor sesgado en el extremo tiene palanca sobre la
   pendiente del último segmento, y puede fabricar una señal de degradación reciente que es artefacto
   del dato.

**La prueba:** correr con la serie completa y con la serie hasta 2024, y comparar el año de quiebre
detectado en los píxeles donde ambas corridas encuentran degradación.

Nótese **qué** se compara: no si las dos corridas detectan lo mismo en 2023–2025 —eso sería
imposible, una tiene un año más— sino si la parte común de la serie se segmenta igual. Por eso solo
entran los quiebres anteriores a 2023.

**Criterio de decisión:** si ≥90 % de los años coinciden (±1), 2025 se conserva. Entre 75 y 90 %,
reportar ambos resultados. Por debajo de 75 %, cerrar la serie en 2024.

In [ ]:
anios_sin_terminal = [a for a in ANIOS if a < ANIO_TERMINAL]
print(f"Corriendo sin {ANIO_TERMINAL} ({len(anios_sin_terminal)} años)...")
lt_sin = correr_landtrendr(serie, anios=anios_sin_terminal)
grad_sin = mapa_cambio(datos_segmentos(lt_sin), UMBRAL_MAG, dur_min=DUR_MIN)

yod_con = grad.select("anio_ini")
yod_sin = grad_sin.select("anio_ini")

# Solo donde AMBAS detectan, y solo quiebres anteriores al corte
ambas = (yod_con.mask().And(yod_sin.mask())
         .And(yod_con.lt(ANIO_CORTE_COMPARACION))
         .And(MASCARA_FINAL.unmask(0)))
coincide = yod_con.subtract(yod_sin).abs().lte(1).updateMask(ambas.selfMask())

print("\nComparando:")
est = evaluar_serie([km2_de(ambas.selfMask()), km2_de(coincide.selfMask())],
                    ["detectan ambas", "coinciden ±1 año"])

pct = est[1] / est[0] * 100 if est[0] else float("nan")

estabilidad = pd.DataFrame([{
    "serie_completa": f"{ANIO_INI}-{ANIO_FIN}",
    "serie_recortada": f"{ANIO_INI}-{ANIO_TERMINAL - 1}",
    "corte_comparacion": ANIO_CORTE_COMPARACION,
    "km2_detectan_ambas": round(est[0], 3),
    "km2_coinciden": round(est[1], 3),
    "pct_coincidencia": round(pct, 1),
}])
estabilidad.to_csv(SALIDA / "estabilidad_2025.csv", index=False)

print(f"\nCONTROL DE ESTABILIDAD · quiebres anteriores a {ANIO_CORTE_COMPARACION}")
print("=" * 66)
print(f"  Superficie donde ambas corridas detectan : {est[0]:7.2f} km²")
print(f"  Donde el año coincide (±1)               : {est[1]:7.2f} km²  ({pct:.1f} %)")
print("=" * 66)
if pct >= 90:
    print(f"✓ La segmentación previa es estable. {ANIO_TERMINAL} se conserva en la serie.")
    print("  Aun así, los quiebres detectados EN el año terminal siguen siendo los menos")
    print("  confiables: no hay observaciones posteriores que los confirmen.")
elif pct >= 75:
    print("~ Estabilidad parcial. Reportar ambos resultados y declararlo en el informe.")
else:
    print(f"✗ {ANIO_TERMINAL} mueve la segmentación previa. Cerrar la serie en "
          f"{ANIO_TERMINAL - 1}:")
    print(f"  poner ANIO_FIN = {ANIO_TERMINAL - 1} en el notebook 05 y volver a ejecutar.")

print("\nGuardado en", SALIDA / "estabilidad_2025.csv")

## 5 · Control 4 · ¿El ajuste es creíble?

Dos miradas al mismo asunto: cuánto se parece la polilínea de LandTrendr a la serie real.

**La distribución del RMSE** dice cuánto se desvía el ajuste en promedio. Pero ojo con la
interpretación: **un RMSE bajo no significa un buen resultado.** Un modelo sobresegmentado se pega
mejor a la serie —más segmentos, menos residuo— y a la vez detecta peor el cambio real. El RMSE
sirve para marcar píxeles poco confiables, no para evaluar el método.

**Las trayectorias** son la mirada que de verdad informa. Superponer la serie original y la ajustada
en píxeles con degradación detectada muestra si la polilínea sigue la forma real o si inventó un
quiebre donde solo había ruido. Es lo que ningún número resume.

In [ ]:
p = con_reintentos(lambda: rmse.updateMask(MASCARA_FINAL).reduceRegion(
    reducer=ee.Reducer.percentile([10, 25, 50, 75, 90, 95]).combine(
        ee.Reducer.mean(), sharedInputs=True),
    geometry=aoi, scale=ESCALA_STATS, maxPixels=1e10, bestEffort=True,
    tileScale=TILE_SCALE).getInfo())

dist = pd.DataFrame([{k.replace("rmse_", ""): round(v, 5) for k, v in p.items()}])
dist.to_csv(SALIDA / "distribucion_rmse.csv", index=False)
display(dist)
print("Guardado en", SALIDA / "distribucion_rmse.csv")
print()
print("Recordatorio: el notebook 05 marca confianza 1/2/3 con los percentiles 75 y 90.")
print(f"  confianza 1 (buena) : rmse <= {p['rmse_p75']:.4f}")
print(f"  confianza 2 (media) : {p['rmse_p75']:.4f} < rmse <= {p['rmse_p90']:.4f}")
print(f"  confianza 3 (baja)  : rmse > {p['rmse_p90']:.4f}")
print()
print("Un RMSE bajo NO es señal de buen resultado: un modelo sobresegmentado baja el")
print("RMSE y empeora la detección. Sirve para marcar píxeles, no para evaluar el método.")

In [ ]:
# Mapas de control
def miniatura(img, vis, ancho=500):
    from PIL import Image
    import requests, io
    url = img.getThumbURL({**vis, "region": aoi, "dimensions": ancho, "format": "png"})
    return np.array(Image.open(io.BytesIO(requests.get(url, timeout=240).content)))


fig, ejes = plt.subplots(1, 3, figsize=(16, 5.5))
ejes[0].imshow(miniatura(grad.select("anio_ini").updateMask(MASCARA_FINAL),
                         {"min": ANIO_INI, "max": ANIO_FIN,
                          "palette": ["#2166ac", "#67a9cf", "#f7f7f7", "#ef8a62", "#b2182b"]}))
ejes[0].set_title("Año de inicio de la degradación")

ejes[1].imshow(miniatura(grad.select("magnitud").updateMask(MASCARA_FINAL),
                         {"min": UMBRAL_MAG, "max": UMBRAL_MAG * 4,
                          "palette": ["#fee5d9", "#fcae91", "#fb6a4a", "#de2d26", "#a50f15"]}))
ejes[1].set_title("Magnitud de la pérdida")

ejes[2].imshow(miniatura(RESULTADO.select("confianza").updateMask(MASCARA_FINAL),
                         {"min": 1, "max": 3, "palette": ["#1a9850", "#fee08b", "#d73027"]}))
ejes[2].set_title("Confianza (verde alta, rojo baja)")

for e in ejes:
    e.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Trayectorias: serie original vs ajuste, en píxeles con degradación detectada
N_MUESTRA = 6
PUNTOS_MANUALES = []     # ej. [(-72.05, -13.55)] si el muestreo automático falla

# La serie original y la ajustada salen del MISMO array: es exactamente lo que vio el
# algoritmo, y no hay que recalcular nada de los compuestos.
lt_base = correr_landtrendr(serie)
fit = (lt_base.select("LandTrendr").arraySlice(0, 2, 3)
       .arrayProject([1]).arrayFlatten([[str(a) for a in ANIOS]]))
orig = (lt_base.select("LandTrendr").arraySlice(0, 1, 2)
        .arrayProject([1]).arrayFlatten([[f"o{a}" for a in ANIOS]]))


def buscar_puntos():
    """Coordenadas de píxeles con degradación. Se muestrea a ESCALA_STATS: a 30 m sobre
    todo el AOI el muestreo desborda la memoria."""
    if PUNTOS_MANUALES:
        return [tuple(x) for x in PUNTOS_MANUALES]
    fc = (grad.select("magnitud").updateMask(MASCARA_FINAL)
          .sample(region=aoi, scale=ESCALA_STATS, numPixels=120, seed=7,
                  dropNulls=True, geometries=True, tileScale=TILE_SCALE))
    return [tuple(f["geometry"]["coordinates"])
            for f in con_reintentos(lambda: fc.limit(N_MUESTRA).getInfo())["features"]]


def trayectoria(lon, lat):
    """En dos consultas separadas en vez de una de 33 bandas: baja el pico de memoria."""
    pt = ee.Geometry.Point([lon, lat])
    a = con_reintentos(lambda: fit.reduceRegion(
        ee.Reducer.first(), pt, ESCALA, tileScale=TILE_SCALE).getInfo())
    b = con_reintentos(lambda: orig.addBands(
        grad.select(["anio_ini", "anio_fin", "magnitud"])).reduceRegion(
        ee.Reducer.first(), pt, ESCALA, tileScale=TILE_SCALE).getInfo())
    return {**a, **b}


try:
    coords = buscar_puntos()
    print(f"{len(coords)} píxeles de muestra con degradación detectada\n")

    fig, ejes = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
    for k, (lon, lat) in enumerate(coords[:6]):
        d = trayectoria(lon, lat)
        eje = ejes[k // 3][k % 3]
        eje.plot(ANIOS, [d.get(str(a)) for a in ANIOS], "o-", color="#c62828", lw=2, ms=4,
                 label="ajustado (LandTrendr)")
        eje.plot(ANIOS, [d.get(f"o{a}") for a in ANIOS], "s--", color="#90a4ae", lw=1, ms=3,
                 alpha=0.8, label="original")
        if d.get("anio_ini"):
            eje.axvspan(d["anio_ini"], d["anio_fin"], alpha=0.15, color="#c62828")
            eje.set_title(f"{d['anio_ini']:.0f}–{d['anio_fin']:.0f} · "
                          f"mag {d['magnitud']:.3f}", fontsize=9)
        eje.grid(alpha=0.3)
        if k == 0:
            eje.legend(fontsize=8)

    plt.suptitle(f"Trayectorias de {INDICE_PRIMARIO}: original vs ajuste de LandTrendr")
    plt.tight_layout()
    plt.show()
    print("Qué mirar: la polilínea roja debe seguir la forma de la serie gris sin inventar")
    print("quiebres donde la original solo tiene ruido.")

except Exception as e:
    print("No se pudieron trazar las trayectorias:", repr(e)[:180])
    print()
    print("Es un problema de memoria, no del método. Dos salidas:")
    print(f"  1. Subir TILE_SCALE (ahora {TILE_SCALE}) a 8 o 16 en la configuración.")
    print("  2. Poner coordenadas a mano en PUNTOS_MANUALES, leídas del mapa de arriba.")
    print()
    print("Es control visual: los tres controles cuantitativos ya están guardados en CSV.")

## 6 · Veredicto

In [ ]:
print("=" * 78)
print("RESUMEN · notebook 06 · control de la detección")
print("=" * 78)
print(f"Resultado controlado : {ASSET_RESULTADO.split('/')[-1]}")
print(f"Corrida del          : {meta.get('fecha_corrida')}")
print(f"Parámetros           : maxSegments={PARAMS['maxSegments']}, "
      f"umbral={UMBRAL_MAG:.4f}, duración mínima={DUR_MIN}")
print()
print("CONTROL 1 · ¿detecta de más?")
print(f"  Tasa dentro del pajonal : {t_dentro:.2f} %")
print(f"  Tasa fuera              : {t_fuera:.2f} %")
print(f"  Razón                   : {razon:.2f}×   "
      f"{'OK' if razon >= 2 else 'JUSTO' if razon >= 1.2 else 'FALLA'}")
print()
print("CONTROL 2 · ¿sobresegmenta?")
for _, f in sens.iterrows():
    marca = "  <<< en uso" if f["maxSegments"] == PARAMS["maxSegments"] else ""
    print(f"  maxSegments {int(f['maxSegments'])}: dentro {f['tasa_dentro_%']:5.2f} % · "
          f"fuera {f['tasa_fuera_%']:5.2f} % · razón {f['razon_dentro_fuera']:.2f}×{marca}")
print()
print(f"CONTROL 3 · ¿{ANIO_TERMINAL} distorsiona?")
print(f"  Coincidencia del año de quiebre : {pct:.1f} %   "
      f"{'OK' if pct >= 90 else 'PARCIAL' if pct >= 75 else 'FALLA'}")
print()
print("CONTROL 4 · ¿el ajuste es creíble?")
print(f"  RMSE mediana : {p['rmse_p50']:.4f}   p90 : {p['rmse_p90']:.4f}")
print("  Las trayectorias hay que mirarlas, no hay número que las resuma.")
print()
print("ARCHIVOS GENERADOS")
for n in ["control_dentro_fuera.csv", "sensibilidad_maxsegments.csv",
          "estabilidad_2025.csv", "distribucion_rmse.csv"]:
    r = SALIDA / n
    print(f"  {'✓' if r.exists() else '·'} {r}")
print()
print("-" * 78)
print("QUÉ SIGNIFICA PASAR ESTOS CONTROLES")
print("-" * 78)
print("Significa COHERENCIA INTERNA: el método no detecta indiscriminadamente, los")
print("parámetros no sobresegmentan y el año terminal no distorsiona lo anterior.")
print()
print("NO significa exactitud. Para eso hacen falta datos de referencia independientes:")
print("muestreo estratificado e interpretación visual, que es el notebook 09. Un")
print("resultado puede pasar todos estos controles y seguir siendo inexacto; pero si")
print("no los pasa, no vale la pena validarlo.")
print()
print("SIGUIENTE · notebook 07 · tipo de cambio")
print("  Leer la dirección de los 6 índices acompañantes en cada vértice (las bandas")
print("  pre/post del resultado) y cruzar con las transiciones de MapBiomas y con el")
print("  asset de quema del notebook 03.")
print("=" * 78)